# 03 - Fine-tune on Colab (free GPU)

Runs the SFT pipeline on a Colab T4 instead of your laptop, with checkpoints
saved to Google Drive so a disconnect doesn't lose progress.

**Before running:** set the runtime to GPU via **Runtime -> Change runtime type -> Hardware accelerator: GPU**, then run the cells top to bottom.

**If the session disconnects mid-training:** re-run cells 1-4 (GPU, clone, install, Drive) and then jump to the **Resume** cell in section 6.

## 1. Confirm a GPU is attached

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - set Runtime -> GPU")

## 2. Clone the repo

In [ ]:
# Safe to re-run: always start from /content, clone only if missing,
# then cd to an ABSOLUTE path so repeated runs never nest folders.
%cd /content
![ -d LLM-from-scratch ] || git clone https://github.com/siddharth0998/LLM-from-scratch.git
%cd /content/LLM-from-scratch

## 3. Install dependencies

Colab already ships torch; we just add the project deps.

In [ ]:
!pip install -q tiktoken transformers pyyaml

## 4. Persist checkpoints to Google Drive

Colab wipes local disk on disconnect. We mount Drive and point the config's
`checkpoint_path` there, so `gpt2-sft.pth` and `last_gpt2-sft.pth` survive and
training can be resumed.

In [ ]:
import yaml
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

CKPT_DIR = '/content/drive/MyDrive/gpt2-sft'
Path(CKPT_DIR).mkdir(parents=True, exist_ok=True)

# Point the SFT config's checkpoint_path at Drive.
cfg_path = 'configs/sft.yaml'
cfg = yaml.safe_load(open(cfg_path))
cfg['checkpoint_path'] = f'{CKPT_DIR}/gpt2-sft.pth'
yaml.safe_dump(cfg, open(cfg_path, 'w'), sort_keys=False)
print('checkpoint_path ->', cfg['checkpoint_path'])

## 5. Prepare the data

Start with a subset for a quick first run. Remove `--limit` for the full 52k.

In [ ]:
!python scripts/prepare_sft_data.py --limit 5000

### Smoke test (optional but recommended)

Overfits a handful of examples; the loss should drop toward ~0.

In [ ]:
!python scripts/train_sft.py --smoke

## 6. Fine-tune

Reads `configs/sft.yaml`, trains, and saves the best model to Drive.
A resumable snapshot (`last_gpt2-sft.pth`) is written to Drive after each epoch.

In [ ]:
!python scripts/train_sft.py

### Resume (run this instead of the cell above if the session dropped)

Loads `last_gpt2-sft.pth` from Drive and continues from the next epoch with the
optimizer state and LR schedule restored. Safe to run even if no snapshot exists.

In [ ]:
!python scripts/train_sft.py --resume

### Loss curves

Plot train/val loss from the training run. The history file is saved next to
the checkpoint on Drive.

In [ ]:
import json, yaml
from pathlib import Path
import matplotlib.pyplot as plt

ckpt = yaml.safe_load(open('configs/sft.yaml'))['checkpoint_path']
history_path = str(Path(ckpt).with_name('history.json'))
h = json.load(open(history_path))

plt.figure(figsize=(8, 4))
plt.plot(h['train_losses'], label='train')
plt.plot(h['val_losses'], label='val')
plt.xlabel('Eval step')
plt.ylabel('Loss')
plt.title('SFT train/val loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Quick chat test with the fine-tuned model

In [ ]:
import sys; sys.path.insert(0, "src")
import yaml, torch
from gpt2_chatbot.model import GPTModel, get_config
from gpt2_chatbot.tokenizer import get_tokenizer, text_to_token_ids, token_ids_to_text, get_eot_id
from gpt2_chatbot.inference import generate
from gpt2_chatbot.data import render_for_inference
from gpt2_chatbot.training import load_checkpoint

ckpt_path = yaml.safe_load(open("configs/sft.yaml"))["checkpoint_path"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = get_config("gpt2-small (124M)", context_length=1024, qkv_bias=True, drop_rate=0.0)
model = GPTModel(cfg)
load_checkpoint(ckpt_path, model, device=device)
model.to(device).eval()
tok = get_tokenizer()

prompt = render_for_inference([{"role": "user", "content": "Give three tips for staying healthy."}])
ids = text_to_token_ids(prompt, tok).to(device)
out = generate(model, ids, max_new_tokens=120, context_size=cfg["context_length"],
               top_k=40, temperature=0.8, eos_id=get_eot_id(tok))
print(token_ids_to_text(out, tok)[len(prompt):])

## 7b. Evaluate on the held-out TEST set

The honest final score: test loss on 2,601 examples the model never saw,
plus a few sample generations vs the reference answers. Run this once, after
training is done.

In [ ]:
# Uses the fine-tuned checkpoint from configs/sft.yaml (your Drive path).
!python scripts/evaluate.py --num-samples 5

## 8. Checkpoint location

The trained model already lives in your Google Drive at
`MyDrive/gpt2-sft/gpt2-sft.pth`, so there's nothing extra to download - just
grab it from Drive. To pull it into the Colab session instead, uncomment below.

In [ ]:
# from google.colab import files
# files.download(yaml.safe_load(open('configs/sft.yaml'))['checkpoint_path'])